# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~2 h on 8×A10G)

---

In Part 3 we turned each card's history into a flat token stream. This notebook pretrains the foundation model on those streams — a Llama causal decoder learning next-token prediction — with Ray Train running the training loop across the cluster. It ends with the trained checkpoint and a HuggingFace export on shared storage: the model every later part uses.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd
import logging
import ray
ray.init(ignore_reinit_error=True, logging_level=logging.ERROR,
         runtime_env={"working_dir": DEMO_ROOT,
                      # torch's native JIT wants a C compiler the workers don't have
                      "env_vars": {"TORCH_DISABLE_NATIVE_JIT": "1"}})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = 8×A10G, ~2h
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Pretrain by predicting the next token

We pretrain the model the way a language model learns text: **next-token prediction** over each card's flat token stream. Given the history so far, the decoder predicts the next token; every position contributes a gradient, so one packed 4096-token sequence yields ~4,000 self-supervised predictions. No labels are involved — the transactions supervise themselves, which is what lets a foundation model learn from far more data than the scarce fraud labels alone.

The model is a **Llama causal decoder** (`src/model.py`, built from `transformers` — GQA, SwiGLU, RMSNorm, RoPE), sized per scale (`full` ≈ 29M params, matching NVIDIA's blueprint). "Causal" means each position attends only to earlier ones — the prediction at every position uses only the past. The hidden state at the final position is what Part 5 extracts as the embedding (following NVIDIA's protocol, each transaction is embedded on its own there).

## The training loop is plain PyTorch — Ray Train scales it

The function that does the work (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, forward, backward, step — with an AdamW optimizer on a **warmup + cosine-decay** learning-rate schedule (schedule, betas, and weight-decay come from the scale config, matching NVIDIA's recipe: lr 2e-4, β₂ 0.95, weight-decay 0.077). Ray Train wraps it with the distributed machinery you'd otherwise hand-write — it launches the workers, shards the dataset across them (`get_dataset_shard` + `iter_torch_batches`), wraps the model in DDP (`prepare_model`), and persists checkpoints to shared storage.

The payoff is the `ScalingConfig`: moving from one CPU worker at `mini` to 8 GPU workers at `full` is a one-line change (`num_workers`, `use_gpu`), not a rewrite of the loop.

`mini` trains a 2-layer model for two CPU epochs in minutes — it proves the loop, not the model. At `full` (8 epochs over the 64,335-sequence corpus, ~16,000 optimizer steps on 8×A10G) budget about two hours; that run reaches perplexity ~1.7.

In [2]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # the per-worker PyTorch loop
import numpy as np
import math

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]
vocab_path = os.path.join(paths["nvcorpus"], "vocab.json")

# The corpus from Part 3 is ids.npy / attn.npy (NVIDIA-tokenizer output). Load it as a Ray
# Dataset of (input_ids, attention_mask) rows — the same columns train_func expects.
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"))
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"))
train_ds = (ray.data.from_numpy(ids).rename_columns({"data": "input_ids"})
            .zip(ray.data.from_numpy(attn).rename_columns({"data": "attention_mask"})))
n_seqs = int(ids.shape[0])

# The cosine LR schedule steps once per optimizer step, so it needs the total step count
# up front: (sequences / workers / batch) * epochs.
steps_per_epoch = max(1, math.ceil(n_seqs / pt["num_workers"] / pt["batch_size"]))
total_steps = steps_per_epoch * pt["epochs"]
warmup_steps = int(pt.get("warmup_ratio", 0.0) * total_steps)
print(f"pretrain (causal LM): {n_seqs:,} sequences · global batch {pt['batch_size'] * pt['num_workers']} "
      f"· ~{steps_per_epoch} steps/epoch × {pt['epochs']} = ~{total_steps:,} optimizer steps "
      f"(warmup {warmup_steps})")

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": vocab_path, "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        # AdamW betas/weight-decay + warmup/cosine LR schedule (from the scale config).
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    # The one line that moves laptop -> cluster: number of workers, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name=f"transaction_fm_pretrain_{SCALE}",      # scale-specific → no cross-scale resume clash
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy weights to the canonical path for Part 5
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

(autoscaler +12s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +12s) [autoscaler] [8cpu-32gb] Attempting to add 1 node to the cluster (increasing from 0 to 1).


(autoscaler +47s) [autoscaler] [8cpu-32gb|m5.2xlarge] [us-west-2a] [on-demand] Launched 1 instance.


(ndarray_to_block pid=2919, ip=10.0.3.184) /tmp/ray/session_2026-07-15_14-32-25_091862_3103/runtime_resources/pip/7bda41e4671c1f0c7eee392064887c47d3f9e183/virtualenv/lib/python3.12/site-packages/cudf/utils/gpu_utils.py:162: UserWarning: No NVIDIA GPU detected
(ndarray_to_block pid=2919, ip=10.0.3.184)   warnings.warn("No NVIDIA GPU detected")


pretrain (causal LM): 8 sequences · global batch 8 · ~1 steps/epoch × 2 = ~2 optimizer steps (warmup 0)


(TrainController pid=47781) A run snapshot was found in storage folder at: '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini'
(TrainController pid=47781) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=47781) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini'.
(TrainController pid=47781) Requesting resources: {'CPU': 1} * 1
(TrainController pid=47781) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=47781) Attempting to start training worker group of size 1 with the following resources: [{'CPU': 1}] * 1


...

(SplitCoordinator pid=48100) ✔️  Dataset train_163_1 execution finished in 0.06 seconds
(SplitCoordinator pid=48100) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=48100) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>


(RayTrainWorker pid=3023, ip=10.0.3.184) [pretrain] epoch 2/2  lm_loss=8.668  ppl=5816.5  lr=3.00e-04
(RayTrainWorker pid=3023, ip=10.0.3.184) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini/checkpoint_2026-07-15_16-13-09.398616)
(RayTrainWorker pid=3023, ip=10.0.3.184) Reporting training result 6: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini/checkpoint_2026-07-15_16-13-09.398616), metrics={'epoch': 1, 'lm_loss': 8.668452262878418, 'perplexity': 5816.489981794646, 'lr': 0.0003}, validation=False)


(TrainController pid=47781) [State Transition] RUNNING -> SHUTTING_DOWN.


done — final lm_loss 8.668, perplexity 5816.5 -> /mnt/cluster_storage/transaction-fm/model/mini/


(TrainController pid=47781) [State Transition] SHUTTING_DOWN -> FINISHED.


## Reading the metrics

Two numbers summarize the run: **causal-LM loss** (cross-entropy) and its exponent, **perplexity** — the effective number of tokens the model is choosing between at each position. Guessing uniformly over the 6,251-token vocabulary would sit at perplexity 6,251; the `full` pretrain ends near **1.7**, fewer than two effective choices per token. That low a number is less surprising than it looks: several of the 12 fields (customer id, card index, state) repeat on nearly every transaction, so the model learns them immediately — the remaining uncertainty is concentrated in the informative fields like amount, merchant, and time. Loss should fall epoch over epoch and settle as the cosine schedule decays the learning rate.

At `mini` (two CPU epochs, a tiny model, 8 sequences) the absolute value means little — it proves the loop runs. The number that matters is the downstream fraud comparison in Part 6, built on the `full` GPU pretrain.

In [3]:
m = result.metrics
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")

final causal-LM loss 8.668   ·   perplexity 5816.5


## Export the decoder for embedding

Part 5 embeds transactions by running them through this decoder with NVIDIA's `HuggingFaceDecoderInference`, which loads a model via `AutoModelForCausalLM.from_pretrained`. So we save our trained checkpoint as a HuggingFace directory (the inner `LlamaForCausalLM`).

In [4]:
import torch
from src.model import build_model

# Export the trained decoder as a HuggingFace directory so Part 5's embedder can load it
# (NVIDIA's HuggingFaceDecoderInference calls AutoModelForCausalLM.from_pretrained). We load our
# TransactionFM checkpoint and save its inner LlamaForCausalLM (m.lm) in HF format.
ck = paths["checkpoint"]
with open(os.path.join(ck, "model_config.json")) as f:
    mc = json.load(f)
m = build_model(vocab_path=os.path.join(ck, "vocab.json"), arch=mc["arch"], max_len=mc["max_len"])
sd = torch.load(os.path.join(ck, "model.pt"), map_location="cpu")
sd = sd["model_state_dict"] if isinstance(sd, dict) and "model_state_dict" in sd else sd
missing, unexpected = m.load_state_dict(sd, strict=False)
os.makedirs(paths["hf"], exist_ok=True)
m.lm.save_pretrained(paths["hf"])
print(f"exported HF decoder -> {paths['hf']}  (state_dict: missing {len(missing)}, unexpected {len(unexpected)})")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

exported HF decoder -> /mnt/cluster_storage/transaction-fm/model_hf/mini/  (state_dict: missing 0, unexpected 0)


## Scaling factors

Training cost is arithmetic: tokens × epochs × model size sets the work, and workers divide the wall-clock. At `full` that is 64,335 sequences × 4096 tokens × 8 epochs through the 29M-parameter decoder — ~16,000 optimizer steps, about two hours on 8×A10G. DDP scales the global batch with the worker count (each worker steps on its own data shard and the gradients average), so eight workers run an epoch in roughly an eighth of the time.

Memory is the constraint that actually bit. The loss needs logits for every position — a (batch × 4096 × 6,251) tensor — and at batch 8 per worker that tensor alone overflowed the A10G's 15.6 GiB. `full` runs batch 4 with gradient checkpointing, which keeps the peak near 9 GiB; the fix lives in the scale config, not the training loop. If the model itself ever outgrows one GPU, `use_fsdp` shards it across workers — at 29M parameters it doesn't need to.

The GPU workers exist for the two hours of this stage and scale back to zero afterward; no other data stage holds them.

## Takeaways

The foundation model exists now: a Llama causal decoder pretrained by next-token prediction over the tokenized corpus, checkpointed to shared storage, and exported as a HuggingFace directory for Part 5. The training loop itself is plain PyTorch — Ray Train supplied the worker launch, dataset sharding, DDP, and checkpoint persistence — and `ScalingConfig` is the only thing that changes between one CPU worker at `mini` and eight GPUs at `full`.

This is also the one stage NVIDIA's repo doesn't run: their notebook trains a 30-step demonstration and downloads the real weights. Here the ~16,000-step run is the notebook — and the Part 1 results table already showed what eight real epochs are worth.

---

## Next

**Part 5 — Batch embedding extraction**: run the pretrained decoder over every transaction — CPU workers tokenize and stream batches to GPU actors for the forward pass, one embedding per transaction.